In [1]:
import pandas as pd

In [2]:
!pip install openpyxl


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [3]:
df= pd.read_excel('Medical_Data_For_Fine_Tuning.xlsx')

In [4]:
df.head()

,Case_ID,Input_Narrative,Structured_Output_JSON
0,PV-REAL-001,A 54-year-old male treated with Atorvastatin f...,"{""seriousness"": true, ""meddra_terms"": [""Myalgi..."
1,PV-REAL-002,A 61-year-old female with type 2 diabetes on M...,"{""seriousness"": true, ""meddra_terms"": [""Somnol..."
2,PV-REAL-003,A 45-year-old male experienced acute tongue sw...,"{""seriousness"": true, ""meddra_terms"": [""Angioe..."
3,PV-REAL-004,A 28-year-old female patient developed anaphyl...,"{""seriousness"": true, ""meddra_terms"": [""Anaphy..."
4,PV-REAL-005,A 68-year-old male on chronic Warfarin therapy...,"{""seriousness"": true, ""meddra_terms"": [""Hematu..."


In [5]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [ ]:
import os
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.0.0" # AMD ROCm hardware spoof
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from trl import SFTTrainer

model_id = "Qwen/Qwen2.5-3B-Instruct"

# 1. Initialize 16-bit precision model & tokenizer
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 2. Configure Standard PEFT LoRA Adapters
peft_config = LoraConfig(
    r=16, lora_alpha=16, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.gradient_checkpointing_enable() # Drastically cuts VRAM usage

# 3. Import and Parse your Excel Sheets
df = pd.read_excel("Medical_Data_For_Fine_Tuning.xlsx")
dataset = Dataset.from_pandas(df)

# 4. Map Rows directly into Qwen Chat Templates
def format_prompts(examples):
    texts = []
    for inp, out in zip(examples["Input_Narrative"], examples["Structured_Output_JSON"]):
        messages = [
            {"role": "system", "content": "Extract PV data into a single-line flattened JSON string."},
            {"role": "user", "content": str(inp)},
            {"role": "assistant", "content": str(out)}
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)

# 5. Define Hugging Face Native Training Hyperparameters
training_args = TrainingArguments(
    output_dir="hf_outputs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=dtype == torch.float16,
    bf16=dtype == torch.bfloat16,
    logging_steps=1,
    report_to="none"
)

# 6. Initialize SFT Engine and Execute Train loop
trainer = SFTTrainer(
    model=model, train_dataset=dataset, dataset_text_field="text",
    max_seq_length=2048, args=training_args
)
trainer.train()

`torch_dtype` is deprecated! Use `dtype` instead!


In [7]:
import os
# ❌ REMOVED: HSA_OVERRIDE_GFX_VERSION (Let ROCm natively target the MI300X CDNA3 architecture)
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
import pandas as pd
from peft import LoraConfig, get_peft_model
from datasets import Dataset
# Remove TrainingArguments from transformers, import SFTConfig from trl instead
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig

model_id = "Qwen/Qwen2.5-3B-Instruct"

# 1. Load with Native FlashAttention-2 
# The MI300X natively excels at bfloat16 and hardware-accelerated FlashAttention
dtype = torch.bfloat16

print("🚀 Loading base model with Native FlashAttention-2 onto MI300X...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=dtype,
    attn_implementation="flash_attention_2", # 🔥 CRITICAL: Leverages MI300X hardware attention kernels
    device_map="cuda:0"                       # Maps directly to your massive HBM3 pool
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 2. Configure PEFT LoRA Adapters
peft_config = LoraConfig(
    r=16, lora_alpha=16, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)

# ❌ REMOVED: model.gradient_checkpointing_enable() 
# Checkpointing trades compute time to save VRAM. With 192GB of VRAM running a tiny 3B model, 
# saving VRAM is pointless. Removing this eliminates a massive computational penalty.

# 3. Import Data
df = pd.read_excel("Medical_Data_For_Fine_Tuning.xlsx")
dataset = Dataset.from_pandas(df)

# 4. Map Rows directly into Qwen Chat Templates
def format_prompts(examples):
    texts = []
    for inp, out in zip(examples["Input_Narrative"], examples["Structured_Output_JSON"]):
        messages = [
            {"role": "system", "content": "Extract PV data into a single-line flattened JSON string."},
            {"role": "user", "content": str(inp)},
            {"role": "assistant", "content": str(out)}
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)

# 5. Define Enterprise-Grade SFT Hyperparameters
training_args = SFTConfig(
    output_dir="hf_outputs",
    per_device_train_batch_size=16,   # Saturation for MI300X
    gradient_accumulation_steps=1,    
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    bf16=True,                        # Pure native bfloat16 math
    logging_steps=1,
    dataloader_num_workers=4,         
    report_to="none",
    
    # 🔥 CRITICAL FIXES FOR MODERN TRL VERSIONS:
    dataset_text_field="text",        # Moved from SFTTrainer to SFTConfig
    max_length=2048                   # Renamed from max_seq_length and moved here
)

# 6. Initialize SFT Engine and Run
print("🔥 Running training loop at maximum hardware capability...")
trainer = SFTTrainer(
    model=model, 
    train_dataset=dataset, 
    args=training_args                # SFTConfig handles everything seamlessly now
)
trainer.train()

🚀 Loading base model with Native FlashAttention-2 onto MI300X...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

/tmp/ipykernel_324/2661587569.py:60: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


🔥 Running training loop at maximum hardware capability...


Adding EOS to train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,3.760800
2,3.768000
3,3.945800
4,3.646300
5,3.347100
6,2.821900
7,2.544900
8,2.299200
9,2.169500
10,1.950500


TrainOutput(global_step=60, training_loss=1.2227276474237443, metrics={'train_runtime': 83.6017, 'train_samples_per_second': 11.483, 'train_steps_per_second': 0.718, 'total_flos': 2698577202708480.0, 'train_loss': 1.2227276474237443, 'epoch': 6.0})

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-3B-Instruct"
adapter_dir = "hf_outputs/checkpoint-60" 

print("⏳ Loading base Qwen model onto clean MI300X memory...")
dtype = torch.bfloat16
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=dtype,
    attn_implementation="flash_attention_2", 
    device_map="cuda:0"
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

print("⚡ Merging fine-tuned LoRA adapters...")
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval() 

# Test medical narrative
test_narrative = (
    "A 62-year-old female patient with a history of chronic plaque psoriasis was started on "
    "Secukinumab (Cosentyx) 300mg weekly. Following the fourth loading dose, she experienced a sudden "
    "onset of severe watery diarrhea, intense abdominal cramping, head spinning and a low-grade fever (38.1°C)."
)

messages = [
    {"role": "system", "content": "Extract PV data into a single-line flattened JSON string."},
    {"role": "user", "content": test_narrative}
]

formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda:0")

print("\n🔮 Executing generation pass...")
with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=256, 
        do_sample=False,              # Greedy decoding for exact, repeatable data extraction
        eos_token_id=tokenizer.eos_token_id
    )

generated_ids = outputs[0][inputs.input_ids.shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("\n🟢 FINE-TUNED MODEL OUTPUT:")
print("=" * 80)
print(response)
print("=" * 80)

⏳ Loading base Qwen model onto clean MI300X memory...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

⚡ Merging fine-tuned LoRA adapters...

🔮 Executing generation pass...

🟢 FINE-TUNED MODEL OUTPUT:
{"seriousness": true, "meddra_terms": ["Diarrhea", "Abdominal pain", "Vertigo"], "labeling_status": ["Diarrhea: Listed", "Abdominal pain: Listed", "Vertigo: Listed"], "naranjo_score": 7, "rationale": "Immune-mediated enterocolitis secondary to monoclonal antibody binding to interleukin-17 receptors."}


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Configuration
base_model_id = "Qwen/Qwen2.5-3B-Instruct"
adapter_dir = "hf_outputs/checkpoint-60" 

# 🔥 TODO: Replace with your Hugging Face username and desired model name
hf_repository_id = "Bokkisam-Rohit24/Qwen2.5-3B-Medical-PV-Extract"
hf_write_token = "<Our-HF-Token>"

print("⏳ Loading base model and tokenizer onto MI300X...")
dtype = torch.bfloat16
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=dtype,
    device_map="cuda:0"
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print("⚡ Attaching your fine-tuned LoRA adapters...")
model = PeftModel.from_pretrained(base_model, adapter_dir)

print("🧩 Merging adapter layers into a single standalone model...")
# This permanently fuses the LoRA weights into the base architecture matrix
merged_model = model.merge_and_unload()

print(f"🚀 Uploading fully merged model to Hugging Face Hub under: {hf_repository_id}")

# Push the weights
merged_model.push_to_hub(
    repo_id=hf_repository_id, 
    token=hf_write_token,
    private=True  # 🔒 Keeps your clinical model private. Change to False if you want it public.
)

# Push the tokenizer config so chat templates work out-of-the-box anywhere
tokenizer.push_to_hub(
    repo_id=hf_repository_id, 
    token=hf_write_token
)

print("\n🎉 Success! Your model is live on Hugging Face and ready to pull down to your laptop.")

⏳ Loading base model and tokenizer onto MI300X...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

⚡ Attaching your fine-tuned LoRA adapters...
🧩 Merging adapter layers into a single standalone model...
🚀 Uploading fully merged model to Hugging Face Hub under: Bokkisam-Rohit24/Qwen2.5-3B-Medical-PV-Extract


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


🎉 Success! Your model is live on Hugging Face and ready to pull down to your laptop.
